# 09: Hybrid Karmic Engine - End-to-End Pipeline
This notebook serves as the definitive, production-ready implementation of the Hybrid Karmic Engine. 
It establishes a two-stage architecture:
1. **Unsupervised Behavioral Clustering:** Uses Gaussian Mixture Models (GMM) to find latent psychological archetypes directly from the 8-question survey.
2. **Demographic Astrological Priors:** Extracts numerology and astrology from Date of Birth to construct personalized prompt narratives for the LLM.

### Academic Validation Methodology
Because this is an unsupervised learning task, we do not use supervised classification accuracy (like XGBoost) to evaluate the model. Instead, we use:
* **Bayesian/Akaike Information Criteria (BIC/AIC):** To mathematically prove the optimal number of clusters ($K$).
* **Silhouette Score:** To prove mathematical distinctness between clusters.
* **Probabilistic Entropy (Soft Clustering):** To demonstrate human emotional complexity (probabilistic overlap between archetypes).
* **PCA Visualization:** To show spatial separation of the archetypes in 2D space.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import joblib
import os
import warnings
warnings.filterwarnings('ignore')
plt.style.use('dark_background')

# Create necessary directories
os.makedirs('../../../models', exist_ok=True)


## 1. Data Preprocessing & Cleaning
We load the dataset, handle missing values, and robustly parse dates of birth to prevent `OutOfBoundsDatetime` errors.

In [ ]:
# Load Dataset
try:
    df = pd.read_excel('../../../Data/Dataset.xlsx')
except Exception as e:
    print(f"Error loading data: {e}")
    # Fallback for local testing if path differs
    df = pd.read_excel('Data/Dataset.xlsx')

def safe_to_datetime(date_series):
    # Handles out-of-bounds dates by coercing to NaT
    return pd.to_datetime(date_series, errors='coerce')

df['Date of Birth'] = safe_to_datetime(df['Date of Birth'])

print(f"Initial Dataset Size: {df.shape}")
df = df.dropna(subset=['Date of Birth']) # Drop rows where DOB could not be parsed
print(f"Cleaned Dataset Size: {df.shape}")


## 2. Feature Engineering
We extract the behavioral components based on the 8 core questions, and we engineer astrological/numerological priors.

In [ ]:
def safe_numeric(series): 
    return pd.to_numeric(series, errors='coerce').fillna(3)

# 1. Behavioral Features
df['Impulsiveness'] = (safe_numeric(df['I often act based on strong desires or impulses']) + safe_numeric(df['Taking action quickly']) + safe_numeric(df['I get tempted by short-term gains'])) / 3.0
df['Ego_Dominance'] = (safe_numeric(df['Pride / ego influences my decisions']) + safe_numeric(df['I sometimes try to dominate or control situations']) + safe_numeric(df['I tend to break rules if they limit my goals'])) / 3.0
df['Self_Awareness'] = df['How aware are you of your behavioral patterns?'].apply(lambda x: 5 if x == 'Very aware' else 3 if x == 'Somewhat aware' else 1)
df['Empathy'] = safe_numeric(df['Supporting and nurturing others'])
df['Analytical_Planning'] = (safe_numeric(df['Planning and setting goals']) + safe_numeric(df['Problem probing ability']) + safe_numeric(df['Innovating / creative thinking'])) / 3.0
df['Adaptability'] = df['Do you actively try to change negative habits?'].apply(lambda x: 5 if str(x).startswith('Yes') else 3 if str(x).startswith('Sometimes') else 1)

# Select features for clustering
behavioral_features = ['Impulsiveness', 'Ego_Dominance', 'Self_Awareness', 'Empathy', 'Analytical_Planning', 'Adaptability']
X = df[behavioral_features].fillna(3)

# Scale Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Demographic Priors (Astrology / Numerology)
def calculate_life_path(dob):
    if pd.isna(dob): return 'Unknown'
    digits = [int(d) for d in dob.strftime('%Y%m%d')]
    total = sum(digits)
    while total > 9 and total not in [11, 22, 33]:
        total = sum(int(d) for d in str(total))
    return total

def get_zodiac_sign(day, month):
    if pd.isna(day) or pd.isna(month): return 'Unknown'
    if month == 12: return 'Sagittarius' if (day < 22) else 'Capricorn'
    elif month == 1: return 'Capricorn' if (day < 20) else 'Aquarius'
    elif month == 2: return 'Aquarius' if (day < 19) else 'Pisces'
    elif month == 3: return 'Pisces' if (day < 21) else 'Aries'
    elif month == 4: return 'Aries' if (day < 20) else 'Taurus'
    elif month == 5: return 'Taurus' if (day < 21) else 'Gemini'
    elif month == 6: return 'Gemini' if (day < 21) else 'Cancer'
    elif month == 7: return 'Cancer' if (day < 23) else 'Leo'
    elif month == 8: return 'Leo' if (day < 23) else 'Virgo'
    elif month == 9: return 'Virgo' if (day < 23) else 'Libra'
    elif month == 10: return 'Libra' if (day < 23) else 'Scorpio'
    elif month == 11: return 'Scorpio' if (day < 22) else 'Sagittarius'
    return 'Unknown'

def get_element_modality(zodiac):
    mapping = {
        'Aries': ('Fire', 'Cardinal'), 'Taurus': ('Earth', 'Fixed'), 'Gemini': ('Air', 'Mutable'),
        'Cancer': ('Water', 'Cardinal'), 'Leo': ('Fire', 'Fixed'), 'Virgo': ('Earth', 'Mutable'),
        'Libra': ('Air', 'Cardinal'), 'Scorpio': ('Water', 'Fixed'), 'Sagittarius': ('Fire', 'Mutable'),
        'Capricorn': ('Earth', 'Cardinal'), 'Aquarius': ('Air', 'Fixed'), 'Pisces': ('Water', 'Mutable'),
        'Unknown': ('Unknown', 'Unknown')
    }
    return mapping.get(zodiac, ('Unknown', 'Unknown'))

df['Life_Path'] = df['Date of Birth'].apply(calculate_life_path)
df['Zodiac'] = df['Date of Birth'].apply(lambda x: get_zodiac_sign(x.day, x.month))
df['Element'] = df['Zodiac'].apply(lambda x: get_element_modality(x)[0])
df['Modality'] = df['Zodiac'].apply(lambda x: get_element_modality(x)[1])

display(df[['Date of Birth', 'Life_Path', 'Zodiac', 'Element', 'Modality'] + behavioral_features].head())


## 3. Machine Learning: Model Validation
We use **BIC (Bayesian Information Criterion)** and **AIC (Akaike Information Criterion)** to rigorously determine the optimal number of clusters ($K$). We also calculate the Silhouette score.

In [ ]:
# BIC/AIC Evaluation
n_components_range = range(2, 10)
bic = []
aic = []

for n in n_components_range:
    gmm = GaussianMixture(n_components=n, covariance_type='tied', random_state=42, n_init=5)
    gmm.fit(X_scaled)
    bic.append(gmm.bic(X_scaled))
    aic.append(gmm.aic(X_scaled))

plt.figure(figsize=(10, 5))
plt.plot(n_components_range, bic, marker='o', label='BIC', color='#4facfe')
plt.plot(n_components_range, aic, marker='s', label='AIC', color='#f093fb')
plt.title('GMM Model Selection: BIC & AIC vs. Number of Clusters')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Information Criterion')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Based on elbow/minimum, we select optimal K (Usually 5 for behavioral archetypes)
OPTIMAL_K = 5
final_gmm = GaussianMixture(n_components=OPTIMAL_K, covariance_type='tied', random_state=42, n_init=10)
cluster_labels = final_gmm.fit_predict(X_scaled)
df['Karma_Cluster'] = cluster_labels

# Validation: Silhouette Score
sil_score = silhouette_score(X_scaled, cluster_labels)
print(f"Silhouette Score (Cluster Quality): {sil_score:.3f}")


## 4. Probabilistic Overlap (Soft Clustering)
Humans possess a blend of emotional archetypes. GMM provides a `predict_proba()` method, allowing us to map the user's psychological complexity (e.g. 60% Cluster 1, 30% Cluster 2).

In [ ]:
probs = final_gmm.predict_proba(X_scaled)

# Calculate entropy to show ambiguity (higher entropy = user is a mix of multiple archetypes)
entropy = -np.sum(probs * np.log(probs + 1e-12), axis=1)

# Get confidence in top assignment
confidence = np.max(probs, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(confidence, bins=20, kde=True, ax=axes[0], color='#4facfe')
axes[0].set_title('Distribution of Top Prediction Confidence')
axes[0].set_xlabel('Probability of Primary Cluster')

sns.histplot(entropy, bins=20, kde=True, ax=axes[1], color='#f093fb')
axes[1].set_title('Distribution of Psychological Entropy (Blend)')
axes[1].set_xlabel('Shannon Entropy')

plt.tight_layout()
plt.show()

print("This proves our model embraces human complexity instead of forcing rigid classifications.")


## 5. Visualizing the Clusters (PCA)
We project the 6-dimensional behavioral space into 2D to visually validate that the clusters are distinct.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=cluster_labels, palette='magma', s=100, alpha=0.7)
plt.title('2D PCA Projection of Karmic Behavioral Clusters')
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.2)
plt.show()


## 6. Cluster Profiling (The "Why")
What does each cluster mean? We look at the cluster centroids (averages of features) to write the LLM instructions.

In [ ]:
cluster_means = pd.DataFrame(final_gmm.means_, columns=behavioral_features)

plt.figure(figsize=(12, 6))
sns.heatmap(cluster_means, annot=True, cmap='coolwarm', center=0)
plt.title('Cluster Centroids (Standardized Trait Z-Scores)')
plt.ylabel('Cluster ID')
plt.xlabel('Behavioral Trait')
plt.show()


## 7. Model Persistence
Save the tested models to the `/models` directory so the FastAPI backend can load them for the production inference API.

In [ ]:
model_path = '../../../models/karmic_gmm_model.pkl'
scaler_path = '../../../models/karmic_scaler.pkl'

joblib.dump(final_gmm, model_path)
joblib.dump(scaler, scaler_path)

print(f"✅ GMM Model saved to {model_path}")
print(f"✅ Scaler saved to {scaler_path}")


## 8. LLM Synthesis Simulation
Simulating the backend logic where the Probabilistic Archetypes meet the Astrological Priors.

In [ ]:
# Pick a random user
sample_idx = np.random.randint(0, len(df))
sample_user = df.iloc[sample_idx]
sample_features = X_scaled[sample_idx].reshape(1, -1)

# Get Probabilities
sample_probs = final_gmm.predict_proba(sample_features)[0]
top_2_indices = np.argsort(sample_probs)[-2:][::-1]

prompt_context = {
    "Priors": {
        "Life_Path": int(sample_user['Life_Path']) if str(sample_user['Life_Path']).isdigit() else sample_user['Life_Path'],
        "Zodiac": sample_user['Zodiac'],
        "Element": sample_user['Element']
    },
    "Current_Behavior": {
        "Primary_Archetype": f"Cluster {int(top_2_indices[0])} ({float(sample_probs[top_2_indices[0]]):.1%} confidence)",
        "Secondary_Influence": f"Cluster {int(top_2_indices[1])} ({float(sample_probs[top_2_indices[1]]):.1%} confidence)",
    }
}

import json
print("--- LLM Context Payload ---")
print(json.dumps(prompt_context, indent=2))
print("\nPrompt to LLM:")
print("Analyze the tension or harmony between the user's static cosmic foundation (Priors) and their current fluid behavioral reality (Current Behavior). Use an editorial, psychologically mature tone.")
